# Modélisation du risque incendie en assurance agricole

Ce notebook présente une approche fréquence–coût pour estimer la prime pure. Il s'inscrit dans un positionnement de **Data Analyst**, de **chargé d'études statistiques** et de **modélisation actuarielle**.

> Les données utilisées ici sont entièrement synthétiques. Les données originales du challenge ne sont pas redistribuées.


## 1. Problème métier

La prime pure est modélisée comme le produit de la fréquence attendue des sinistres et de leur coût moyen attendu. La charge attendue tient ensuite compte de l'exposition du contrat.

Cette séparation permet d'identifier si un profil est risqué en raison d'une fréquence plus élevée, d'une sévérité plus forte, ou des deux.


In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "synthetic_agricultural_portfolio.csv"
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"

df = pd.read_csv(DATA)
print(f"{len(df):,} contrats synthétiques et {df.shape[1]} variables")
df.head()


## 2. Contrôles de qualité

Les contrôles portent sur les dimensions, les doublons, les valeurs manquantes et la cohérence des variables cibles.


In [ ]:
quality = pd.Series({
    "observations": len(df),
    "duplicate_policy_ids": df["policy_id"].duplicated().sum(),
    "missing_values": int(df.isna().sum().sum()),
    "negative_claim_counts": int((df["claim_count"] < 0).sum()),
    "negative_costs": int((df["total_claim_cost_eur"] < 0).sum()),
})
quality.to_frame("value")


## 3. Analyse descriptive

La fréquence est fortement concentrée sur zéro, tandis que les coûts positifs présentent une asymétrie à droite. Ces propriétés motivent l'utilisation de modèles adaptés aux données d'assurance.


In [ ]:
display(Image(filename=str(FIGURES / "target_distributions.png")))


## 4. Modélisation hors échantillon

Le script reproductible compare :

- GLM Poisson et Gradient Boosting Poisson pour la fréquence ;
- GLM Tweedie et Random Forest pour le coût moyen.

Il utilise une séparation entraînement–test et applique les transformations au moyen de pipelines afin de limiter les fuites de données.


In [ ]:
# Décommenter pour régénérer toutes les données, figures et sorties.
# %run ../src/generate_synthetic_data.py
# %run ../src/run_analysis.py

metrics = pd.read_csv(RESULTS / "model_metrics.csv")
metrics.round({"MAE": 3, "deviance": 3})


In [ ]:
display(Image(filename=str(FIGURES / "model_comparison.png")))


## 5. Résultat fréquence–coût

Le modèle final combine la meilleure fréquence et la meilleure sévérité. En assurance, il est important d'évaluer l'erreur individuelle, mais aussi la calibration agrégée du portefeuille.


In [ ]:
summary = json.loads((RESULTS / "summary.json").read_text(encoding="utf-8"))
pd.Series(summary, name="value").to_frame()


## 6. Interprétation des facteurs de risque

L'importance par permutation mesure la dégradation de la performance lorsque chaque variable est perturbée. Elle facilite la lecture du modèle, sans établir de relation causale.


In [ ]:
display(Image(filename=str(FIGURES / "frequency_feature_importance.png")))


## Conclusion

L'approche fréquence–coût relie l'analyse de données à une problématique actuarielle concrète. Sur la démonstration synthétique, le GLM Poisson est retenu pour la fréquence et le Random Forest pour le coût selon la déviance. Les résultats ne sont ni ceux du challenge ni une tarification réelle : ils valident uniquement la reproductibilité du pipeline public.
